# Potentiostat Serial Smoke Tests

Purpose: verify that 4 potentiostats on `/dev/ttyACM0..3` are visible and can be opened reliably.

This notebook avoids hardware-specific protocol commands. It only performs safe serial I/O checks.

## What these tests confirm

1. Linux sees all expected ttyACM devices
2. Python can open/close each port
3. Permissions are sufficient (no permission denied)
4. Passive read works (if device streams data)
5. Repeated connect/disconnect is stable

## What these tests do not confirm
- Electrochemistry protocol correctness
- Measurement quality
- Device-specific command semantics

In [26]:
from pathlib import Path
import glob
import os
import platform
import subprocess
import sys
import time

import serial
import serial.tools.list_ports

print('Python   :', sys.version.split()[0])
print('Platform :', platform.platform())

Python   : 3.12.3
Platform : Linux-6.17.0-14-generic-x86_64-with-glibc2.39


In [27]:
EXPECTED_PORTS = [f'/dev/ttyACM{i}' for i in range(4)]
DEFAULT_BAUD = 115200
OPEN_TIMEOUT = 1.0
PASSIVE_READ_SECONDS = 1.5

EXPECTED_PORTS

['/dev/ttyACM0', '/dev/ttyACM1', '/dev/ttyACM2', '/dev/ttyACM3']

In [28]:
detected_acm = sorted(glob.glob('/dev/ttyACM*'))
detected_usb = sorted(glob.glob('/dev/ttyUSB*'))
print('Detected /dev/ttyACM*:', detected_acm)
print('Detected /dev/ttyUSB*:', detected_usb)

missing = [p for p in EXPECTED_PORTS if p not in detected_acm]
if missing:
    print('Missing expected ports:', missing)
else:
    print('All expected ttyACM0..3 are present.')

Detected /dev/ttyACM*: ['/dev/ttyACM0', '/dev/ttyACM1', '/dev/ttyACM2', '/dev/ttyACM3']
Detected /dev/ttyUSB*: []
All expected ttyACM0..3 are present.


In [29]:
ports = list(serial.tools.list_ports.comports())
print(f'pyserial sees {len(ports)} total ports')
for p in ports:
    print('-' * 70)
    print('device      :', p.device)
    print('description :', p.description)
    print('hwid        :', p.hwid)
    print('vid/pid     :', p.vid, p.pid)
    print('manufacturer:', p.manufacturer)
    print('product     :', p.product)
    print('serial_no   :', p.serial_number)
    print('location    :', p.location)

pyserial sees 36 total ports
----------------------------------------------------------------------
device      : /dev/ttyS31
description : n/a
hwid        : n/a
vid/pid     : None None
manufacturer: None
product     : None
serial_no   : None
location    : None
----------------------------------------------------------------------
device      : /dev/ttyS30
description : n/a
hwid        : n/a
vid/pid     : None None
manufacturer: None
product     : None
serial_no   : None
location    : None
----------------------------------------------------------------------
device      : /dev/ttyS29
description : n/a
hwid        : n/a
vid/pid     : None None
manufacturer: None
product     : None
serial_no   : None
location    : None
----------------------------------------------------------------------
device      : /dev/ttyS28
description : n/a
hwid        : n/a
vid/pid     : None None
manufacturer: None
product     : None
serial_no   : None
location    : None
---------------------------------------

In [30]:
def open_close_test(port: str, baud: int = DEFAULT_BAUD, timeout: float = OPEN_TIMEOUT):
    result = {'port': port, 'opened': False, 'error': None}
    try:
        with serial.Serial(port, baudrate=baud, timeout=timeout) as ser:
            result['opened'] = ser.is_open
    except Exception as exc:
        result['error'] = repr(exc)
    return result

open_results = [open_close_test(p) for p in EXPECTED_PORTS]
open_results

[{'port': '/dev/ttyACM0', 'opened': True, 'error': None},
 {'port': '/dev/ttyACM1', 'opened': True, 'error': None},
 {'port': '/dev/ttyACM2', 'opened': True, 'error': None},
 {'port': '/dev/ttyACM3', 'opened': True, 'error': None}]

In [31]:
def passive_read_test(port: str, baud: int = DEFAULT_BAUD, timeout: float = OPEN_TIMEOUT, wait_s: float = PASSIVE_READ_SECONDS):
    result = {
        'port': port,
        'opened': False,
        'bytes_read': 0,
        'ascii_preview': '',
        'hex_preview': '',
        'error': None,
    }
    try:
        with serial.Serial(port, baudrate=baud, timeout=timeout) as ser:
            result['opened'] = True
            ser.reset_input_buffer()
            time.sleep(wait_s)
            waiting = ser.in_waiting
            payload = ser.read(waiting or 256)
            result['bytes_read'] = len(payload)
            result['ascii_preview'] = payload.decode('utf-8', errors='replace')[:200]
            result['hex_preview'] = payload[:64].hex(' ')
    except Exception as exc:
        result['error'] = repr(exc)
    return result

passive_results = [passive_read_test(p) for p in EXPECTED_PORTS]
passive_results

[{'port': '/dev/ttyACM0',
  'opened': True,
  'bytes_read': 0,
  'ascii_preview': '',
  'hex_preview': '',
  'error': None},
 {'port': '/dev/ttyACM1',
  'opened': True,
  'bytes_read': 0,
  'ascii_preview': '',
  'hex_preview': '',
  'error': None},
 {'port': '/dev/ttyACM2',
  'opened': True,
  'bytes_read': 0,
  'ascii_preview': '',
  'hex_preview': '',
  'error': None},
 {'port': '/dev/ttyACM3',
  'opened': True,
  'bytes_read': 0,
  'ascii_preview': '',
  'hex_preview': '',
  'error': None}]

In [32]:
def reconnect_stability_test(port: str, cycles: int = 10, baud: int = DEFAULT_BAUD, timeout: float = OPEN_TIMEOUT):
    errors = []
    for i in range(cycles):
        try:
            with serial.Serial(port, baudrate=baud, timeout=timeout):
                pass
        except Exception as exc:
            errors.append((i, repr(exc)))
    return {'port': port, 'cycles': cycles, 'error_count': len(errors), 'errors': errors}

stability_results = [reconnect_stability_test(p, cycles=10) for p in EXPECTED_PORTS]
stability_results

[{'port': '/dev/ttyACM0', 'cycles': 10, 'error_count': 0, 'errors': []},
 {'port': '/dev/ttyACM1', 'cycles': 10, 'error_count': 0, 'errors': []},
 {'port': '/dev/ttyACM2', 'cycles': 10, 'error_count': 0, 'errors': []},
 {'port': '/dev/ttyACM3', 'cycles': 10, 'error_count': 0, 'errors': []}]

In [33]:
def probe_baud(port: str, baud_list=(9600, 19200, 57600, 115200, 230400), timeout: float = OPEN_TIMEOUT):
    rows = []
    for b in baud_list:
        try:
            with serial.Serial(port, baudrate=b, timeout=timeout) as ser:
                ser.reset_input_buffer()
                time.sleep(0.3)
                waiting = ser.in_waiting
                payload = ser.read(waiting or 128)
                rows.append({'port': port, 'baud': b, 'opened': True, 'bytes_read': len(payload), 'error': None})
        except Exception as exc:
            rows.append({'port': port, 'baud': b, 'opened': False, 'bytes_read': 0, 'error': repr(exc)})
    return rows

baud_results = []
for p in EXPECTED_PORTS:
    baud_results.extend(probe_baud(p))

baud_results

[{'port': '/dev/ttyACM0',
  'baud': 9600,
  'opened': True,
  'bytes_read': 0,
  'error': None},
 {'port': '/dev/ttyACM0',
  'baud': 19200,
  'opened': True,
  'bytes_read': 0,
  'error': None},
 {'port': '/dev/ttyACM0',
  'baud': 57600,
  'opened': True,
  'bytes_read': 0,
  'error': None},
 {'port': '/dev/ttyACM0',
  'baud': 115200,
  'opened': True,
  'bytes_read': 0,
  'error': None},
 {'port': '/dev/ttyACM0',
  'baud': 230400,
  'opened': True,
  'bytes_read': 0,
  'error': None},
 {'port': '/dev/ttyACM1',
  'baud': 9600,
  'opened': True,
  'bytes_read': 0,
  'error': None},
 {'port': '/dev/ttyACM1',
  'baud': 19200,
  'opened': True,
  'bytes_read': 0,
  'error': None},
 {'port': '/dev/ttyACM1',
  'baud': 57600,
  'opened': True,
  'bytes_read': 0,
  'error': None},
 {'port': '/dev/ttyACM1',
  'baud': 115200,
  'opened': True,
  'bytes_read': 0,
  'error': None},
 {'port': '/dev/ttyACM1',
  'baud': 230400,
  'opened': True,
  'bytes_read': 0,
  'error': None},
 {'port': '/dev/tt

In [34]:
# Optional Linux-only diagnostics
for cmd in [
    ['bash', '-lc', 'ls -l /dev/ttyACM* /dev/ttyUSB* 2>/dev/null || true'],
    ['bash', '-lc', 'ls -l /dev/serial/by-id 2>/dev/null || true'],
    ['bash', '-lc', 'groups'],
]:
    print('=' * 80)
    print('CMD:', ' '.join(cmd))
    out = subprocess.run(cmd, capture_output=True, text=True)
    print(out.stdout if out.stdout else '<no stdout>')
    if out.stderr:
        print('stderr:', out.stderr)

CMD: bash -lc ls -l /dev/ttyACM* /dev/ttyUSB* 2>/dev/null || true
crw-rw-rw-+ 1 root dialout 166, 0 Mar 10 16:55 /dev/ttyACM0
crw-rw-rw-+ 1 root dialout 166, 1 Mar 10 16:55 /dev/ttyACM1
crw-rw-rw-+ 1 root dialout 166, 2 Mar 10 16:55 /dev/ttyACM2
crw-rw-rw-+ 1 root dialout 166, 3 Mar 10 16:55 /dev/ttyACM3

CMD: bash -lc ls -l /dev/serial/by-id 2>/dev/null || true
total 0
lrwxrwxrwx 1 root root 13 Mar 10 16:55 usb-STMicroelectronics_STM32_Virtual_ComPort_206630914232-if00 -> ../../ttyACM2
lrwxrwxrwx 1 root root 13 Mar 10 16:55 usb-STMicroelectronics_STM32_Virtual_ComPort_2084307C4232-if00 -> ../../ttyACM3
lrwxrwxrwx 1 root root 13 Mar 10 16:55 usb-STMicroelectronics_STM32_Virtual_ComPort_208430804232-if00 -> ../../ttyACM0
lrwxrwxrwx 1 root root 13 Mar 10 16:55 usb-STMicroelectronics_STM32_Virtual_ComPort_208430864232-if00 -> ../../ttyACM1

CMD: bash -lc groups
gavin adm dialout cdrom sudo dip plugdev users bluetooth lpadmin docker libvirt ollama kvm



## Physical verification (multimeter procedure)

The potentiostat is an STM32-based electrochemical instrument. It controls voltage between three electrodes — **WE** (working), **RE** (reference), **CE** (counter) — and measures the resulting current through a sense resistor.

### Before you connect electrodes

1. **Power on** — plug in all 4 boards via USB. Confirm `/dev/ttyACM0..3` appear (above). A status LED on the board (if present) should be solid or blinking.
2. **Switch state** — on power-up the firmware sets the switch **OFF** (WE/CE disconnected). This is safe.
3. **Multimeter: VREF pin** — probe the board's VREF pad and GND. Should read **≈ 1.65 V** (midpoint of the 3.3 V ADC supply). If you see 0 V or 3.3 V the board may not be powered or the reference is damaged.
4. **Multimeter: WE → CE continuity** — with switch OFF, you should see **open circuit** (no beep). This confirms the protection switch is working.

### With a test resistor (no chemistry needed)

5. Connect a known resistor (e.g. **1 kΩ**) between **WE** and **CE**, leave **RE** floating (or tie to WE).
6. Run the `set_voltage_and_read` protocol cell below with a small voltage (e.g. ±0.1 V). You should read back **I ≈ V / R** (e.g. 0.1 V / 1 kΩ = 100 µA). A result within ~5 % confirms the full signal chain is working.

### With real electrodes (chemistry)

7. Place the three electrodes in a known-composition solution (e.g. 5 mM ferricyanide in 0.1 M KCl — a standard CV benchmark).
8. Run a cyclic voltammetry sweep (−0.6 V to +0.6 V, 50 mV/s). You should see a symmetric redox peak pair at ~+0.2 V and ~+0.1 V vs Ag/AgCl.


## Protocol-level verification (firmware alive?)

These cells use the binary serial protocol from `devices/potentiostat/barebones.py` + the command set discovered in the archived driver (`scripts/archived_potentiostat/ps4_ref.py`).

**Protocol framing:** every message is `[device_id: uint8, command: uint8, ...optional data bytes]`.  
**Device ID** is `1` for all units (firmware default). Each unit lives on its own USB serial port so the ID byte is just a framing header.

| Command | Byte | Sends | Receives |
|---|---|---|---|
| `READ_SWITCH` | 17 | `[id, 17]` | 1 byte bool |
| `READ_GAIN` | 16 | `[id, 16, 0]` | 1 byte resistor index |
| `READ_ADC` | 1 | `[id, 1, ch]` | 4 bytes float32 per channel |

A response to any of these = firmware is running.


In [35]:
import struct, numpy as np

# ── Command constants (from ps4_ref.py Commands enum) ─────────────────────────
CMD_READ_SWITCH = 17   # → 1 byte uint8  (0=off, 1=on)
CMD_READ_GAIN   = 16   # → 1 byte uint8  (resistor index 0-4)
CMD_READ_ADC    =  1   # → n_channels × 4 bytes float32

DEVICE_ID = 1          # firmware-side address; all units default to 1

# ── Resistor lookup (for human-readable gain label) ───────────────────────────
GAIN_LABELS = {0: '100 Ω', 1: '1 kΩ', 2: '10 kΩ', 3: '100 kΩ', 4: '1 MΩ'}


def firmware_ping(port: str, device_id: int = DEVICE_ID,
                  baud: int = DEFAULT_BAUD, timeout: float = 2.0) -> dict:
    """Send READ_SWITCH then READ_GAIN; confirm firmware responds to both."""
    result = {
        'port': port,
        'alive': False,
        'switch_on': None,
        'gain': None,
        'gain_label': None,
        'error': None,
    }
    try:
        with serial.Serial(port, baudrate=baud, timeout=timeout) as ser:
            ser.reset_input_buffer()

            # -- READ_SWITCH: [device_id, 17]  → 1 byte
            ser.write(bytes([device_id, CMD_READ_SWITCH]))
            resp = ser.read(1)
            if len(resp) != 1:
                result['error'] = f'READ_SWITCH timed out (got {len(resp)} bytes)'
                return result
            result['switch_on'] = bool(resp[0])

            # -- READ_GAIN: [device_id, 16, 0]  → 1 byte
            ser.write(bytes([device_id, CMD_READ_GAIN, 0]))
            resp = ser.read(1)
            if len(resp) != 1:
                result['error'] = f'READ_GAIN timed out (got {len(resp)} bytes)'
                return result
            g = int(resp[0])
            result['gain'] = g
            result['gain_label'] = GAIN_LABELS.get(g, f'unknown({g})')

            result['alive'] = True
    except Exception as exc:
        result['error'] = repr(exc)
    return result


ping_results = [firmware_ping(p) for p in EXPECTED_PORTS]
for r in ping_results:
    status = '✓ ALIVE' if r['alive'] else '✗ NO RESPONSE'
    print(f"{r['port']}  {status}  switch={'ON' if r['switch_on'] else 'off'}  gain={r['gain_label']}  err={r['error']}")


/dev/ttyACM0  ✓ ALIVE  switch=off  gain=1 MΩ  err=None
/dev/ttyACM1  ✓ ALIVE  switch=off  gain=1 MΩ  err=None
/dev/ttyACM2  ✓ ALIVE  switch=off  gain=1 MΩ  err=None
/dev/ttyACM3  ✓ ALIVE  switch=off  gain=1 MΩ  err=None


In [36]:
# ADC channel IDs (from ps4_ref.py ADC class)
ADC_WE_OUT = 0   # WE–CE potential → converted to current via sense resistor
ADC_RE_OUT = 1   # WE–RE potential (the controlled voltage)
ADC_VREF   = 2   # Internal ADC reference voltage — should be ~1.65 V raw
ADC_TEMP   = 3   # Temperature (raw ADC counts, board-dependent scaling)
ADC_HUMID  = 4   # Humidity    (raw ADC counts, board-dependent scaling)

# Voltage scaling: firmware stores ADC values in the 0–3.3 V range,
# but they represent a ±5 V bipolar signal.  Map: 0 V → -5 V, 3.3 V → +5 V.
def scale_adc(raw_v: float) -> float:
    return float(np.interp(raw_v, [0.0, 3.3], [-5.0, 5.0]))


def read_adc_channels(port: str, channels: list, device_id: int = DEVICE_ID,
                      baud: int = DEFAULT_BAUD, timeout: float = 2.0) -> dict:
    """Send READ_ADC for each channel individually; return raw and scaled values."""
    result = {'port': port, 'readings': {}, 'error': None}
    ch_names = {ADC_WE_OUT: 'WE_OUT', ADC_RE_OUT: 'RE_OUT',
                ADC_VREF: 'VREF', ADC_TEMP: 'TEMP', ADC_HUMID: 'HUMID'}
    try:
        with serial.Serial(port, baudrate=baud, timeout=timeout) as ser:
            ser.reset_input_buffer()
            for ch in channels:
                ser.write(bytes([device_id, CMD_READ_ADC, ch]))
                raw = ser.read(4)          # 1 × float32
                if len(raw) != 4:
                    result['readings'][ch_names.get(ch, ch)] = 'TIMEOUT'
                    continue
                raw_v = struct.unpack('<f', raw)[0]
                scaled_v = scale_adc(raw_v) if ch not in (ADC_TEMP, ADC_HUMID) else raw_v
                result['readings'][ch_names.get(ch, ch)] = {
                    'raw_V': round(raw_v, 4),
                    'scaled_V': round(scaled_v, 4),
                }
    except Exception as exc:
        result['error'] = repr(exc)
    return result


# Read VREF, RE_OUT, TEMP, HUMID on all ports (safe: switch is still OFF)
adc_results = [read_adc_channels(p, [ADC_VREF, ADC_RE_OUT, ADC_TEMP, ADC_HUMID])
               for p in EXPECTED_PORTS]

for r in adc_results:
    print(f"\n{r['port']}  err={r['error']}")
    for ch, val in r['readings'].items():
        print(f"  {ch:<10} {val}")

# Expected with no electrodes connected and switch OFF:
#   VREF  raw ≈ 1.65 V  →  scaled ≈ 0.0 V  (internal ADC midpoint reference)
#   RE_OUT scaled ≈ 0.0 V (floating / pulled to midpoint)
#   TEMP / HUMID: raw ADC counts — non-zero if a sensor is fitted on the board



/dev/ttyACM0  err=SerialException('device reports readiness to read but returned no data (device disconnected or multiple access on port?)')
  VREF       {'raw_V': 0.0331, 'scaled_V': -4.8997}
  RE_OUT     {'raw_V': 3.2886, 'scaled_V': 4.9654}
  TEMP       TIMEOUT

/dev/ttyACM1  err=SerialException('device reports readiness to read but returned no data (device disconnected or multiple access on port?)')
  VREF       {'raw_V': 0.0332, 'scaled_V': -4.8995}
  RE_OUT     {'raw_V': 3.2872, 'scaled_V': 4.9613}
  TEMP       TIMEOUT

/dev/ttyACM2  err=SerialException('device reports readiness to read but returned no data (device disconnected or multiple access on port?)')
  VREF       {'raw_V': 0.0331, 'scaled_V': -4.8996}
  RE_OUT     {'raw_V': 3.2877, 'scaled_V': 4.9628}
  TEMP       TIMEOUT

/dev/ttyACM3  err=SerialException('device reports readiness to read but returned no data (device disconnected or multiple access on port?)')
  VREF       {'raw_V': 0.0331, 'scaled_V': -4.8996}
  RE_OUT

## Success criteria

You are in good shape if:
- all of `/dev/ttyACM0..3` exist
- open/close test has no errors
- reconnect stability test reports `error_count = 0` for each port
- passive read either returns bytes or consistently returns 0 bytes without errors

0 passive bytes can still be normal if the device waits for protocol commands.

## Next step (protocol test)

After these smoke tests pass, add one device-specific command test on a single port (`/dev/ttyACM0`) to verify command/response framing.
Keep that test isolated before parallelizing across 4 devices.